In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = '/home/codespace/.config/gcloud/application_default_credentials.json'

gcs_jar = '/workspaces/etl-zoomcamp-project/lib/gcs-connector-hadoop3-2.2.5.jar'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('my-etl-project') \
    .set("spark.jars", gcs_jar) \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/03/22 17:05:24 WARN Utils: Your hostname, codespaces-b411cf resolves to a loopback address: 127.0.0.1; using 10.0.1.104 instead (on interface eth0)
26/03/22 17:05:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/22 17:05:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
df_gdp = spark.read.csv("gs://mental-health-etl-bucket/raw/gdp_data.csv", header=True)

In [28]:
import pandas as pd

pandas_df = df_gdp.toPandas()
pandas_df.isnull().sum().sum()

np.int64(0)

In [29]:
df_gdp.count()

10134

In [30]:
df_gdp.printSchema()

root
 |-- year: string (nullable = true)
 |-- rank: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gdp: string (nullable = true)
 |-- gdp_percent: string (nullable = true)



In [31]:
countries = [row["country"] for row in df_gdp.select("country").distinct().collect()]
print(countries)

['Chad', 'Micronesia (Federated States of)', 'Paraguay', 'Russia', 'Macao', 'Yemen', 'Senegal', 'Sweden', 'Kiribati', 'Guyana', 'Philippines', 'Eritrea', 'Tonga', 'Djibouti', 'Malaysia', 'Singapore', 'Turkey', 'Malawi', 'Iraq', 'Germany', 'Northern Mariana Islands', 'Comoros', 'Cambodia', 'Afghanistan', "C ô te d'Ivoire", 'Rwanda', 'Jordan', 'Maldives', 'Sudan', 'Palau', 'France', 'Turks and Caicos Islands', 'Greece', 'Sri Lanka', 'Congo (gold)', 'Timor Leste', 'Dominica', 'Algeria', 'Togo', 'Equatorial Guinea', 'Slovakia', 'Argentina', 'Belgium', 'Angola', 'San Marino', 'Ecuador', 'Qatar', 'Lesotho', 'South Sultan', 'Madagascar', 'Albania', 'Finland', 'New Caledonia', 'Nicaragua', 'Myanmar', 'Peru', 'Sierra Leone', 'Benin', 'China', 'India', 'Bahamas', 'Belarus', 'Kuwait', 'Malta', 'American Samoa', 'Somalia', 'Marshall Islands', 'Tuvalu', 'Chile', 'Puerto Rico', 'Tajikistan', 'Cayman Islands', 'Croatia', 'Burundi', 'Nigeria', 'Bolivia', 'Andorra', 'Gabon', 'Italy', 'Suriname', 'Nethe

In [32]:
from pyspark.sql import types
from pyspark.sql.functions import round, col

schema = types.StructType([
    types.StructField('year', types.IntegerType(), True),
    types.StructField('rank', types.IntegerType(), True),
    types.StructField('country', types.StringType(), True),
    types.StructField('state', types.StringType(), True),
    types.StructField('gdp', types.LongType(), True),
    types.StructField('gdp_percent', types.DoubleType(), True)
])


In [41]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("gs://mental-health-etl-bucket/raw/gdp_data.csv")

In [42]:
df.printSchema()

root
 |-- year: integer (nullable = true)
 |-- rank: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gdp: long (nullable = true)
 |-- gdp_percent: double (nullable = true)



In [57]:
df.show()

+----+----+-----------------+-------+------------+-----------+---------+
|year|rank|          country|  state|         gdp|gdp_percent|gdp_short|
+----+----+-----------------+-------+------------+-----------+---------+
|1960|   1|the United States|America|543300000000|    0.46848|   543.3B|
|1960|   2|   United Kingdom| Europe| 73233967692|    0.06315|    73.2B|
|1960|   3|           France| Europe| 62225478000|    0.05366|    62.2B|
|1960|   4|            China|   Asia| 59716467625|    0.05149|    59.7B|
|1960|   5|            Japan|   Asia| 44307342950|    0.03821|    44.3B|
|1960|   6|           Canada|America| 40461721692|    0.03489|    40.5B|
|1960|   7|            Italy| Europe| 40385288344|    0.03482|    40.4B|
|1960|   8|            India|   Asia| 37029883875|    0.03193|    37.0B|
|1960|   9|        Australia|Oceania| 18577668271|    0.01602|    18.6B|
|1960|  10|           Sweden| Europe| 15822585033|    0.01364|    15.8B|
|1960|  11|           Brazil|America| 15165569912| 

In [58]:
df = df.withColumn("gdp_percent", round(col("gdp_percent"), 5))

In [59]:
unique_years = [row['year'] for row in df.select("year").distinct().orderBy("year").collect()]

print(unique_years)

[1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]


In [60]:
from pyspark.sql import functions as F

In [61]:
df = df.withColumn(
    "gdp_short",
    F.when(F.col("gdp") >= 1_000_000_000,
           F.concat(F.round(F.col("gdp")/1_000_000_000,1), F.lit("B")))
    .when(F.col("gdp") >= 1_000_000,
           F.concat(F.round(F.col("gdp")/1_000_000,0), F.lit("M")))
    .otherwise(F.col("gdp"))
)

In [62]:
from pyspark.sql.functions import when, col

df_clean = df.withColumn(
    "country",
    when(col("country") == "C ô te d'Ivoire", "Cote d'Ivoire")
    .when(col("country") == "South Sultan", "South Sudan")
    .when(col("country") == "Fiji Islands", "Fiji")
    .when(col("country") == "Cura ç Ao", "Curacao")
    .when(col("country") == "the United States", "United States")
    .when(col("country") == "Sao Tome and Principe.", "Sao Tome and Principe")
    .when(col("country") == "Columbia", "Colombia")
    .when(col("country") == "Isle of man", "Isle of Man")
    .when(col("country") == "Guinea Bissau", "Guinea-Bissau")
    .when(col("country") == "Czech", "Czech Republic")
    .when(col("country") == "Congo (gold)", "Democratic Republic of Congo")
    .when(col("country") == "Congo (Brazzaville)", "Congo")
    .when(col("country") == "Central Africa", "Central African Republic")
    .otherwise(col("country"))
)

In [65]:
df_clean.filter(col("country") == "Cote d'Ivoire").show()

+----+----+-------------+------+----------+-----------+---------+
|year|rank|      country| state|       gdp|gdp_percent|gdp_short|
+----+----+-------------+------+----------+-----------+---------+
|1960|  65|Cote d'Ivoire|Africa| 546203561|     4.7E-4|   546.0M|
|1961|  64|Cote d'Ivoire|Africa| 618245639|     5.1E-4|   618.0M|
|1962|  67|Cote d'Ivoire|Africa| 645284344|     4.9E-4|   645.0M|
|1963|  61|Cote d'Ivoire|Africa| 761047045|     5.4E-4|   761.0M|
|1964|  58|Cote d'Ivoire|Africa| 921063266|     5.9E-4|   921.0M|
|1965|  64|Cote d'Ivoire|Africa| 919771356|     5.4E-4|   920.0M|
|1966|  62|Cote d'Ivoire|Africa|1024103034|     5.5E-4|     1.0B|
|1967|  64|Cote d'Ivoire|Africa|1082922892|     5.5E-4|     1.1B|
|1968|  63|Cote d'Ivoire|Africa|1281281245|     6.0E-4|     1.3B|
|1969|  63|Cote d'Ivoire|Africa|1361360157|     5.8E-4|     1.4B|
|1970|  67|Cote d'Ivoire|Africa|1455482990|     5.2E-4|     1.5B|
|1971|  67|Cote d'Ivoire|Africa|1584128262|     5.1E-4|     1.6B|
|1972|  68

In [66]:
df_clean.show(10)

+----+----+--------------+-------+------------+-----------+---------+
|year|rank|       country|  state|         gdp|gdp_percent|gdp_short|
+----+----+--------------+-------+------------+-----------+---------+
|1960|   1| United States|America|543300000000|    0.46848|   543.3B|
|1960|   2|United Kingdom| Europe| 73233967692|    0.06315|    73.2B|
|1960|   3|        France| Europe| 62225478000|    0.05366|    62.2B|
|1960|   4|         China|   Asia| 59716467625|    0.05149|    59.7B|
|1960|   5|         Japan|   Asia| 44307342950|    0.03821|    44.3B|
|1960|   6|        Canada|America| 40461721692|    0.03489|    40.5B|
|1960|   7|         Italy| Europe| 40385288344|    0.03482|    40.4B|
|1960|   8|         India|   Asia| 37029883875|    0.03193|    37.0B|
|1960|   9|     Australia|Oceania| 18577668271|    0.01602|    18.6B|
|1960|  10|        Sweden| Europe| 15822585033|    0.01364|    15.8B|
+----+----+--------------+-------+------------+-----------+---------+
only showing top 10 

In [67]:
df_clean.write.mode("overwrite").parquet("gs://mental-health-etl-bucket/processed/gdp_data.parquet")